In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"

FIGURE_DIR = OUTPUT_DIR / "final_publication_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:")
print(FIGURE_DIR)


# ---------------------------------------------------------
# Publication style
# ---------------------------------------------------------

sns.set_theme(
    context="paper",
    style="whitegrid",
    font="DejaVu Sans",
    font_scale=1.05
)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9,
    "legend.title_fontsize": 9.5,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.55,
    "savefig.facecolor": "white",
    "savefig.bbox": "tight"
})


# ---------------------------------------------------------
# Consistent colours
# ---------------------------------------------------------

CATEGORY_COLOURS = {
    "police": "#3B5B92",
    "legal": "#79558C",
    "support_sector": "#2A7F78",
    "health": "#C17C3A",
    "welfare_state": "#6F7D45"
}


# ---------------------------------------------------------
# Export helper
# ---------------------------------------------------------

def save_publication_figure(fig, filename):
    """
    Export the same figure as:
    - high-resolution PNG for Word;
    - PDF and SVG for vector-quality archiving.
    """
    
    png_path = FIGURE_DIR / f"{filename}.png"
    pdf_path = FIGURE_DIR / f"{filename}.pdf"
    svg_path = FIGURE_DIR / f"{filename}.svg"
    
    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    
    print("Saved:")
    print(png_path)
    print(pdf_path)
    print(svg_path)

In [ ]:
# ---------------------------------------------------------
# Figure 1 — Final retrieval-validation precision
# Source: NB03 final validation output
# Analysis unit: manually reviewed KWIC record
# ---------------------------------------------------------

validation_path = OUTPUT_DIR / "validation_summary_v2.xlsx"

assert validation_path.exists(), (
    f"File not found: {validation_path}"
)

validation = pd.read_excel(
    validation_path,
    sheet_name="category_summary"
)

print("Loaded columns:")
print(validation.columns.tolist())

display(validation)

In [ ]:
# ---------------------------------------------------------
# Exact-data verification
# ---------------------------------------------------------

expected_validation = pd.DataFrame({
    "category": [
        "legal",
        "police",
        "support_sector",
        "health",
        "welfare_state"
    ],
    "sampled_n": [53, 53, 58, 56, 55],
    "valid_n": [49, 46, 46, 27, 24],
    "validity_pct": [92.5, 86.8, 79.3, 48.2, 43.6]
})

validation_check = (
    validation[
        ["category", "sampled_n", "valid_n", "validity_pct"]
    ]
    .sort_values("category")
    .reset_index(drop=True)
)

expected_check = (
    expected_validation
    .sort_values("category")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    validation_check,
    expected_check,
    check_dtype=False
)

assert validation["sampled_n"].sum() == 275
assert validation["valid_n"].sum() == 192

print("PASSED: final validation data verified.")
print("Overall:", "192/275 =", f"{192 / 275:.1%}")

In [ ]:
# ---------------------------------------------------------
# Publication-ready horizontal lollipop chart
# ---------------------------------------------------------

plot_data = (
    validation
    .sort_values("validity_pct", ascending=True)
    .copy()
)

category_labels = {
    "legal": "Legal",
    "police": "Police",
    "support_sector": "Support sector",
    "health": "Health",
    "welfare_state": "Welfare / state"
}

plot_data["category_label"] = (
    plot_data["category"].map(category_labels)
)

# Three focus categories use blue;
# lower-precision categories use muted grey.
focus_categories = {
    "legal",
    "police",
    "support_sector"
}

plot_data["colour"] = plot_data["category"].apply(
    lambda x: "#315F85"
    if x in focus_categories
    else "#9AA6AF"
)

fig, ax = plt.subplots(figsize=(8.4, 4.8))

y_positions = np.arange(len(plot_data))

# Horizontal stems
for y, value, colour in zip(
    y_positions,
    plot_data["validity_pct"],
    plot_data["colour"]
):
    ax.hlines(
        y=y,
        xmin=0,
        xmax=value,
        color=colour,
        linewidth=3.2,
        alpha=0.75
    )

# End points
ax.scatter(
    plot_data["validity_pct"],
    y_positions,
    s=105,
    color=plot_data["colour"],
    edgecolor="white",
    linewidth=1.2,
    zorder=3
)

# Exact labels
for y, (_, row) in zip(
    y_positions,
    plot_data.iterrows()
):
    label = (
        f'{row["valid_n"]}/{row["sampled_n"]} '
        f'({row["validity_pct"]:.1f}%)'
    )
    
    ax.text(
        row["validity_pct"] + 2,
        y,
        label,
        va="center",
        ha="left",
        fontsize=9.5,
        color="#252525"
    )

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_data["category_label"])

ax.set_xlim(0, 108)
ax.set_xticks(np.arange(0, 101, 20))
ax.set_xticklabels(
    [f"{x}%" for x in np.arange(0, 101, 20)]
)

ax.set_xlabel("Valid institutional mentions in validation sample")
ax.set_ylabel("")

ax.set_title(
    "Validation Precision of Institutional Keyword Retrieval",
    loc="left",
    pad=14
)

# Overall result as a subtitle
ax.text(
    0,
    1.02,
    "Final validation sample: 192/275 valid records (69.8%)",
    transform=ax.transAxes,
    fontsize=9.5,
    color="#555555"
)

ax.grid(axis="x", visible=True)
ax.grid(axis="y", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(axis="y", length=0)

plt.tight_layout()

save_publication_figure(
    fig,
    "figure_validation_precision_final"
)

plt.show()

## Distinctive TF-IDF Terms

This figure displays the six most distinctive terms for each institutional retrieval category.

Distinctive score:

> Mean TF-IDF in the focal category − mean TF-IDF across the other four categories.

Analysis unit: transcript × retrieval-category document.

Final TF-IDF corpus: N = 627 documents and 2,800 unigram features.

In [ ]:
# ---------------------------------------------------------
# Load final category-distinctive TF-IDF outputs
# Source: Notebook 04, Step 9I
# ---------------------------------------------------------

TFIDF_CATEGORIES = [
    "police",
    "legal",
    "health",
    "support_sector",
    "welfare_state"
]

tfidf_results = {}

for category in TFIDF_CATEGORIES:
    
    file_path = (
        OUTPUT_DIR /
        f"step9_distinctive_terms_{category}.csv"
    )
    
    assert file_path.exists(), (
        f"File not found: {file_path}"
    )
    
    df = pd.read_csv(file_path)
    
    required_columns = {
        "term",
        "category_mean_tfidf",
        "other_categories_mean",
        "distinctive_score"
    }
    
    assert required_columns.issubset(df.columns), (
        f"Missing columns in {category}: "
        f"{required_columns - set(df.columns)}"
    )
    
    assert len(df) == 2800, (
        f"{category}: expected 2,800 features, "
        f"found {len(df)}"
    )
    
    assert df["distinctive_score"].notna().all()
    
    tfidf_results[category] = (
        df.sort_values(
            "distinctive_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

print("Loaded final TF-IDF outputs:")
for category, df in tfidf_results.items():
    print(
        category,
        "| features:",
        len(df),
        "| top term:",
        df.iloc[0]["term"],
        "| score:",
        round(df.iloc[0]["distinctive_score"], 4)
    )

In [ ]:
# ---------------------------------------------------------
# Verify the expected final lexical profiles
# ---------------------------------------------------------

expected_top_terms = {
    "police": [
        "police",
        "report",
        "arrest",
        "officer",
        "call",
        "order"
    ],
    
    "legal": [
        "court",
        "solicitor",
        "legal",
        "judge",
        "justice",
        "criminal"
    ],
    
    "health": [
        "mental",
        "health",
        "counselling",
        "doctor",
        "hospital",
        "disability"
    ],
    
    "support_sector": [
        "refuge",
        "charity",
        "support",
        "helpline",
        "worker",
        "victim"
    ],
    
    "welfare_state": [
        "housing",
        "income",
        "status",
        "rent",
        "probation",
        "council"
    ]
}

for category, expected_terms in expected_top_terms.items():
    
    actual_terms = (
        tfidf_results[category]
        .head(6)["term"]
        .tolist()
    )
    
    assert actual_terms == expected_terms, (
        f"{category} top terms do not match.\n"
        f"Expected: {expected_terms}\n"
        f"Actual:   {actual_terms}"
    )

print("PASSED: final top-six TF-IDF terms verified.")

In [ ]:
# ---------------------------------------------------------
# Compact publication-ready TF-IDF small multiples
# Two columns + centred final panel
# ---------------------------------------------------------

from matplotlib.gridspec import GridSpec

document_counts = {
    "police": 139,
    "legal": 133,
    "health": 136,
    "support_sector": 93,
    "welfare_state": 126
}

category_display = {
    "police": "Police",
    "legal": "Legal",
    "health": "Health",
    "support_sector": "Support sector",
    "welfare_state": "Welfare / state"
}

TOP_N = 5

global_max = max(
    tfidf_results[category]
    .head(TOP_N)["distinctive_score"]
    .max()
    for category in TFIDF_CATEGORIES
)

x_limit = global_max * 1.25

# Four-column grid:
# each panel occupies two columns;
# the final panel occupies the middle two columns.
fig = plt.figure(figsize=(10.2, 7.4))

gs = GridSpec(
    nrows=3,
    ncols=4,
    figure=fig,
    hspace=0.62,
    wspace=0.95
)

axes = {
    "police": fig.add_subplot(gs[0, 0:2]),
    "legal": fig.add_subplot(gs[0, 2:4]),
    "health": fig.add_subplot(gs[1, 0:2]),
    "welfare_state": fig.add_subplot(gs[1, 2:4]),
    "support_sector": fig.add_subplot(gs[2, 1:3])
}

plot_order = [
    "police",
    "legal",
    "health",
    "welfare_state",
    "support_sector"
]

for category in plot_order:
    
    ax = axes[category]
    
    plot_df = (
        tfidf_results[category]
        .head(TOP_N)
        .sort_values(
            "distinctive_score",
            ascending=True
        )
        .copy()
    )
    
    y_positions = np.arange(len(plot_df))
    colour = CATEGORY_COLOURS[category]
    
    ax.barh(
        y_positions,
        plot_df["distinctive_score"],
        height=0.58,
        color=colour,
        alpha=0.88
    )
    
    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["term"],
        fontsize=8.8
    )
    
    # Direct score labels
    for y, value in zip(
        y_positions,
        plot_df["distinctive_score"]
    ):
        ax.text(
            value + global_max * 0.018,
            y,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=8.1,
            color="#252525"
        )
    
    ax.set_title(
        (
            f"{category_display[category]} "
            f"(n = {document_counts[category]})"
        ),
        loc="left",
        fontsize=10.5,
        color=colour,
        pad=5
    )
    
    ax.set_xlim(0, x_limit)
    
    ax.grid(
        axis="x",
        visible=True,
        linewidth=0.5,
        alpha=0.45
    )
    
    ax.grid(
        axis="y",
        visible=False
    )
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    
    ax.tick_params(
        axis="y",
        length=0
    )
    
    ax.tick_params(
        axis="x",
        labelsize=8
    )

# Only bottom panel needs the shared x-axis label
axes["support_sector"].set_xlabel(
    "Distinctive TF-IDF score",
    fontsize=9.5
)

fig.suptitle(
    "Distinctive TF-IDF Terms across Institutional Retrieval Categories",
    x=0.08,
    y=0.985,
    ha="left",
    fontsize=13.5,
    fontweight="bold"
)

fig.text(
    0.08,
    0.952,
    (
        "Top five terms per category; "
        "627 transcript × category documents; "
        "2,800 unigram features"
    ),
    ha="left",
    fontsize=9,
    color="#555555"
)

plt.subplots_adjust(
    top=0.89,
    bottom=0.09,
    left=0.12,
    right=0.97
)

save_publication_figure(
    fig,
    "figure_tfidf_distinctive_profiles_compact_final"
)

plt.show()

## Evaluative Direction by Institutional Category

This figure presents evaluative-direction composition within the final manually annotated sample.

Analysis unit: valid annotated passage.

Final sample: N = 73 valid annotations from 60 unique transcripts.

In [ ]:
# ---------------------------------------------------------
# Load final manually reviewed annotation data
# Source: Notebook 07
# ---------------------------------------------------------

annotation_path = (
    OUTPUT_DIR /
    "step16_formal_annotation_sample_human_reviewed.xlsx"
)

assert annotation_path.exists(), (
    f"File not found: {annotation_path}"
)

annotation = pd.read_excel(annotation_path)

print("Annotation shape:")
print(annotation.shape)

print("\nColumns:")
print(annotation.columns.tolist())

print("\nValid mention values:")
print(annotation["valid_mention"].value_counts(dropna=False))

print("\nEvaluative-direction values:")
print(
    annotation["evaluative_direction"]
    .value_counts(dropna=False)
)

In [ ]:
# ---------------------------------------------------------
# Recover final evaluative-direction table
# ---------------------------------------------------------

# Normalise labels without altering substantive values
annotation["valid_mention"] = (
    annotation["valid_mention"]
    .astype(str)
    .str.strip()
    .str.lower()
)

annotation["category"] = (
    annotation["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

annotation["evaluative_direction"] = (
    annotation["evaluative_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
)

focus_categories = [
    "police",
    "legal",
    "support_sector"
]

valid_annotations = annotation[
    (annotation["valid_mention"] == "yes") &
    (annotation["category"].isin(focus_categories))
].copy()

assert len(valid_annotations) == 73
assert valid_annotations["transcript_id"].nunique() == 60

direction_order = [
    "negative",
    "mixed",
    "neutral_descriptive",
    "positive"
]

direction_counts = pd.crosstab(
    valid_annotations["category"],
    valid_annotations["evaluative_direction"]
)

direction_counts = (
    direction_counts
    .reindex(
        index=focus_categories,
        columns=direction_order,
        fill_value=0
    )
)

display(direction_counts)

expected_direction_counts = pd.DataFrame(
    {
        "negative": [12, 11, 3],
        "mixed": [3, 2, 4],
        "neutral_descriptive": [3, 9, 4],
        "positive": [7, 3, 12]
    },
    index=[
        "police",
        "legal",
        "support_sector"
    ]
)

expected_direction_counts.index.name = "category"
expected_direction_counts.columns.name = (
    "evaluative_direction"
)

pd.testing.assert_frame_equal(
    direction_counts,
    expected_direction_counts,
    check_dtype=False
)

assert direction_counts.sum(axis=1).to_dict() == {
    "police": 25,
    "legal": 25,
    "support_sector": 23
}

assert direction_counts.values.sum() == 73

print("PASSED: final evaluative-direction counts verified.")

In [ ]:
# ---------------------------------------------------------
# Publication-ready 100% stacked bar chart
# ---------------------------------------------------------

direction_proportions = (
    direction_counts
    .div(
        direction_counts.sum(axis=1),
        axis=0
    )
    * 100
)

direction_labels = {
    "negative": "Negative",
    "mixed": "Mixed",
    "neutral_descriptive": "Neutral",
    "positive": "Positive"
}

direction_colours = {
    "negative": "#A64B42",
    "mixed": "#D49A4A",
    "neutral_descriptive": "#B8BDC4",
    "positive": "#2F7F78"
}

category_labels = {
    "police": "Police (n = 25)",
    "legal": "Legal (n = 25)",
    "support_sector": "Support sector (n = 23)"
}

fig, ax = plt.subplots(
    figsize=(8.4, 3.8)
)

left_positions = np.zeros(
    len(direction_proportions)
)

for direction in direction_order:
    
    values = direction_proportions[direction].values
    
    bars = ax.barh(
        y=np.arange(len(direction_proportions)),
        width=values,
        left=left_positions,
        height=0.58,
        color=direction_colours[direction],
        edgecolor="white",
        linewidth=1.0,
        label=direction_labels[direction]
    )
    
    # Percentage labels inside segments
    for row_position, (
        left,
        width
    ) in enumerate(
        zip(left_positions, values)
    ):
        
        if width >= 7.5:
            text_colour = (
                "white"
                if direction in [
                    "negative",
                    "positive"
                ]
                else "#252525"
            )
            
            ax.text(
                left + width / 2,
                row_position,
                f"{width:.1f}%",
                ha="center",
                va="center",
                fontsize=8.7,
                color=text_colour,
                fontweight="bold"
            )
    
    left_positions += values

ax.set_yticks(
    np.arange(len(direction_proportions))
)

ax.set_yticklabels([
    category_labels[category]
    for category in direction_proportions.index
])

ax.set_xlim(0, 100)

ax.set_xticks(
    np.arange(0, 101, 20)
)

ax.set_xticklabels([
    f"{value}%"
    for value in np.arange(0, 101, 20)
])

ax.set_xlabel(
    "Percentage of valid annotated passages"
)

ax.set_ylabel("")

ax.set_title(
    "Evaluative Direction within the Final Annotated Sample",
    loc="left",
    pad=14
)

ax.text(
    0,
    1.03,
    (
        "N = 73 valid passage-level annotations "
        "from 60 unique transcripts"
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=9.3,
    color="#555555"
)

ax.legend(
    title="Evaluative direction",
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.24),
    frameon=False
)

ax.grid(
    axis="x",
    visible=True,
    alpha=0.35
)

ax.grid(
    axis="y",
    visible=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(
    axis="y",
    length=0
)

# Show Police first at the top
ax.invert_yaxis()

plt.tight_layout()

save_publication_figure(
    fig,
    "figure_evaluative_direction_final"
)

plt.show()

In [ ]:
# ============================================================
# Institutional-role profiles by institution
# Final verified data from Notebook07
# N = 73 valid annotations from 60 unique transcripts
# Row denominators: Legal 25, Police 25, Support sector 23
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Exact FINAL VERIFIED counts from Notebook07
role_counts = pd.DataFrame(
    {
        "Protective /\nsupportive": [3, 7, 11],
        "Partial /\nlimited support": [4, 6, 4],
        "Neutral /\ndescriptive": [9, 1, 2],
        "Bureaucratic /\ninaccessible": [4, 1, 3],
        "Absent /\nunavailable": [1, 4, 3],
        "Harmful /\nretraumatising": [3, 4, 0],
        "Dismissive /\ndisbelieving": [1, 2, 0],
    },
    index=["Legal", "Police", "Support sector"]
)

row_n = pd.Series(
    {"Legal": 25, "Police": 25, "Support sector": 23}
)

# Presentation conversion only: within-category row percentages
role_pct = role_counts.div(row_n, axis=0) * 100

# Each cell shows exact count and within-category percentage
annotations = role_counts.copy().astype(str)

for institution in role_counts.index:
    for role in role_counts.columns:
        n = role_counts.loc[institution, role]
        pct = role_pct.loc[institution, role]
        annotations.loc[institution, role] = f"{n}\n({pct:.1f}%)"

sns.set_theme(style="white", font_scale=0.95)

fig, ax = plt.subplots(figsize=(11.2, 3.8))

sns.heatmap(
    role_pct,
    annot=annotations,
    fmt="",
    cmap=sns.light_palette("#315A6B", as_cmap=True),
    vmin=0,
    vmax=50,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={
        "label": "Within-category percentage",
        "shrink": 0.78,
        "pad": 0.02
    },
    annot_kws={
        "fontsize": 9,
        "fontweight": "normal"
    },
    ax=ax
)

ax.set_title(
    "Institutional-role Profiles by Institutional Category",
    fontsize=13,
    fontweight="bold",
    loc="center",
    pad=14
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_yticklabels(
    [f"Legal\n(n = 25)", f"Police\n(n = 25)", f"Support sector\n(n = 23)"],
    rotation=0,
    fontsize=10
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=0,
    ha="center",
    fontsize=8.5
)

ax.tick_params(axis="both", length=0)



plt.tight_layout()

# Use the output folder defined at the start of this notebook
output_path = FIGURE_DIR / "figure_institutional_role_profiles.png"

fig.savefig(
    output_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {output_path}")

In [ ]:
# ============================================================
# Justice-dimension profiles by institution
# Final verified data from Notebook 07
# N = 73 valid annotations from 60 unique transcripts
# Row denominators: Legal 25, Police 25, Support sector 23
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Exact FINAL VERIFIED counts from Notebook07
justice_counts = pd.DataFrame(
    {
        "Safety /\nprevention": [4, 11, 11],
        "Consequences": [9, 5, 1],
        "Connectedness /\nrecovery": [4, 0, 7],
        "Voice": [5, 4, 1],
        "Recognition": [2, 3, 3],
        "Dignity": [1, 2, 0],
    },
    index=["Legal", "Police", "Support sector"]
)

row_n = pd.Series(
    {"Legal": 25, "Police": 25, "Support sector": 23}
)

# Presentation conversion only: within-category percentages
justice_pct = justice_counts.div(row_n, axis=0) * 100

# Cell labels: exact count and row percentage
annotations = justice_counts.copy().astype(str)

for institution in justice_counts.index:
    for dimension in justice_counts.columns:
        n = justice_counts.loc[institution, dimension]
        pct = justice_pct.loc[institution, dimension]
        annotations.loc[institution, dimension] = f"{n}\n({pct:.1f}%)"

sns.set_theme(style="white", font_scale=0.95)

fig, ax = plt.subplots(figsize=(10.2, 3.8))

sns.heatmap(
    justice_pct,
    annot=annotations,
    fmt="",
    cmap=sns.light_palette("#5A4772", as_cmap=True),
    vmin=0,
    vmax=50,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={
        "label": "Within-category percentage",
        "shrink": 0.78,
        "pad": 0.02
    },
    annot_kws={
        "fontsize": 9,
        "fontweight": "normal"
    },
    ax=ax
)

ax.set_title(
    "Justice-dimension Profiles by Institutional Category",
    fontsize=13,
    fontweight="bold",
    loc="center",
    pad=14
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_yticklabels(
    [
        "Legal\n(n = 25)",
        "Police\n(n = 25)",
        "Support sector\n(n = 23)"
    ],
    rotation=0,
    fontsize=10
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=0,
    ha="center",
    fontsize=9
)

ax.tick_params(axis="both", length=0)

plt.tight_layout()

# Use the same output folder as the previous figures
output_path = FIGURE_DIR / "figure_justice_dimension_profiles.png"

fig.savefig(
    output_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {output_path}")

In [ ]:
# ============================================================
# KWIC retrieval volume and transcript coverage
# Final verified output from Notebook 03
# Corpus: 141 analytical transcripts
# Total raw KWIC hits: 8,865
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Exact FINAL VERIFIED values from NB03
kwic_summary = pd.DataFrame(
    {
        "Institution": [
            "Police",
            "Legal",
            "Health",
            "Welfare / state",
            "Support sector"
        ],
        "KWIC hits": [3368, 3098, 1266, 747, 386],
        "Unique transcripts": [139, 133, 136, 126, 93]
    }
)

# Keep categories ordered by raw retrieval volume
kwic_summary = kwic_summary.sort_values(
    "KWIC hits",
    ascending=True
).reset_index(drop=True)

sns.set_theme(style="whitegrid", font_scale=0.95)

fig, (ax1, ax2) = plt.subplots(
    ncols=2,
    figsize=(10.5, 4.3),
    sharey=True,
    gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.08}
)

# ------------------------------------------------------------
# Left panel: raw KWIC hits
# ------------------------------------------------------------

bars = ax1.barh(
    kwic_summary["Institution"],
    kwic_summary["KWIC hits"],
    color="#496D7A",
    height=0.58
)

ax1.set_title(
    "Raw KWIC hits",
    fontsize=11,
    fontweight="bold",
    pad=10
)

ax1.set_xlabel("Number of retrieved KWIC records")
ax1.set_ylabel("")

ax1.set_xlim(0, 3800)

for bar, value in zip(bars, kwic_summary["KWIC hits"]):
    ax1.text(
        value + 55,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        ha="left",
        fontsize=9,
        color="#222222"
    )

# ------------------------------------------------------------
# Right panel: transcript coverage
# ------------------------------------------------------------

ax2.hlines(
    y=kwic_summary["Institution"],
    xmin=0,
    xmax=kwic_summary["Unique transcripts"],
    color="#C5CDD1",
    linewidth=2
)

ax2.scatter(
    kwic_summary["Unique transcripts"],
    kwic_summary["Institution"],
    s=75,
    color="#496D7A",
    edgecolor="white",
    linewidth=0.8,
    zorder=3
)

ax2.set_title(
    "Transcript coverage",
    fontsize=11,
    fontweight="bold",
    pad=10
)

ax2.set_xlabel("Unique transcripts with ≥1 hit")
ax2.set_ylabel("")
ax2.set_xlim(0, 150)

for institution, value in zip(
    kwic_summary["Institution"],
    kwic_summary["Unique transcripts"]
):
    ax2.text(
        value + 3,
        institution,
        f"{value}",
        va="center",
        ha="left",
        fontsize=9,
        color="#222222"
    )

# ------------------------------------------------------------
# Shared academic styling
# ------------------------------------------------------------

fig.suptitle(
    "Institutional KWIC Retrieval Volume and Transcript Coverage",
    fontsize=13,
    fontweight="bold",
    x=0.5,
    y=1.02
)

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color="#E2E5E7", linewidth=0.7)
    ax.tick_params(axis="y", length=0)

plt.tight_layout()

# Save publication-ready asset
output_path = FIGURE_DIR / "figure_kwic_retrieval_and_coverage.png"

fig.savefig(
    output_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {output_path}")